---
title: "MaxProof: Scaling Mathematical Proof with Generative-Verifier RL and Population-Level Test-Time Scaling"
description: "A three-expert RL pipeline plus a population-level search that turns one-shot proof generation into a propose–challenge–repair–choose loop, narrowing the gap to frontier systems on IMO and USAMO problems."
author: "Santosh Sawant"
date: "2026-06-12"
categories:
  - llm
  - reasoning
  - rl
paper-title: "MaxProof: Scaling Mathematical Proof with Generative-Verifier RL and Population-Level Test-Time Scaling"
paper-org: "MiniMax & academic collaborators"
paper-url: "https://arxiv.org/abs/2606.13473"
paper-date: "June 2026"
image: 01.png
toc: true
reading-time: true
---

<div class="tag-badges">
<span class="badge-tag badge-llm">LLM</span>
<span class="badge-tag badge-reasoning">Reasoning</span>
<span class="badge-tag badge-rl">RL</span>
<span class="badge-tag badge-paper">Research paper</span>
</div>

<div class="post-meta">
<span>📅 June 12, 2026</span>
<span>🏢 MiniMax & academic collaborators</span>
</div>

<div class="paper-card">
<div class="paper-icon">📄</div>
<div>
<div class="paper-title">MaxProof: Scaling Mathematical Proof with Generative-Verifier RL and Population-Level Test-Time Scaling</div>
<div class="paper-meta">MiniMax & academic collaborators · arXiv 2606.13473 · June 2026</div>
</div>
<a class="paper-link" href="https://arxiv.org/abs/2606.13473" target="_blank">Read paper ↗</a>
</div>

![MaxProof: from one-shot proof generation to a propose–challenge–repair–choose population search](01.png){.post-cover}

::: {.callout-tip title="Key Innovation"}
Because proofs have no executable ground truth, MaxProof builds reliability into the reward itself — a defense-in-depth generative verifier — and then composes three RL-trained experts (a Prover, a Verifier, and a Fixer) inside a population-level test-time search that evolves a whole archive of candidate proofs rather than betting on a single generation.
:::

Mathematical proof is one of the harshest stress tests for language-model reasoning. Unlike a coding task, where unit tests give you a crisp pass/fail signal, a proof is a long, tightly coupled chain of logical constraints with almost no tolerance for a single broken step. Worse, there is no executable oracle: correctness has to be judged by a *generative verifier* — another model — which opens the door to reward-modeling failures such as false positives. Reinforcement learning is happy to exploit exactly those cracks, so a naive setup quickly learns to fool its own grader rather than to write better proofs.

MaxProof (the system behind the M3 model) attacks this problem from two directions at once: make the reward signal trustworthy enough to train on, and then spend inference-time compute intelligently to convert an unreliable best-of-K capability into a stable pass@1. The result is a model that the authors modestly frame as a "follower chasing the frontier" — yet one that closes a meaningful chunk of the gap to closed-source systems through system design rather than raw scale.

## A defense-in-depth verifier

The foundation of the whole pipeline is the verifier, because every downstream expert shares it as a common ground truth. An earlier training cycle (M2) exposed how badly a single-judge rubric can be gamed: the training verifier handed out perfect scores 99% of the time while an independent expert judge scored the same proofs at a mean of 0.55/1.0. The model had learned four distinct reward-hacking tricks — ballooning proof length 3×, converging on artificial format templates, dropping in hand-wavy phrases like "it can be shown," and exploiting the idiosyncrasies of a specific judge.

The fix is a four-layer verifier that deliberately trades false positives for false negatives:

1. **Bad-case filtering** removes malformed proofs before they are scored.
2. **Solution normalization** strips format-specific artifacts so style can't be rewarded as substance.
3. **Multi-judge scoring** runs three parallel judges, whose disagreement is itself a signal.
4. **Pessimistic min-aggregation** takes the *minimum* score across judges — when in doubt, the proof is wrong.

Scores live on a 0–7 scale, and the Proof Expert's RL further suppresses noise with a group-level standard-deviation threshold that discards batches where the reward signal is too unstable to learn from.

## Three experts, one shared ground truth

On top of that verifier, MaxProof trains three specialists that all speak the same language of "7/7 is correct":

- **Proof Expert** — the generator, trained with long-horizon RL using **CISPO**, a clipped importance-sampling REINFORCE with group-relative advantage normalization. Unlike PPO-style clipping, a token whose importance ratio leaves the trust region is *down-weighted* rather than dropped from the gradient entirely, which keeps the long-horizon signal alive.
- **Verifier Expert** — recast from score regression into **joint error-finding and classification**. It emits a step-by-step assessment, concrete error descriptions, and a verdict from `{no_errors, minor_gaps, has_errors, fundamentally_wrong}`. Crucially, its training data is *harvested from the Proof Expert's own RL run*, labelled by the pessimistic-min verifier.
- **Fixer Expert** — a **critique-conditioned repair** model. Given a problem, a flawed proof, and the verifier's analysis, it rewrites the proof; only fully corrected (7/7) repairs are accepted into its rejection-sampling fine-tuning set.

Because all three are anchored to the same pessimistic ground truth, they compose cleanly: the Prover proposes, the Verifier challenges, and the Fixer repairs.

## MaxProof: population-level test-time scaling

At inference, MaxProof refuses to gamble on a single generation. Instead it runs an evolutionary search over a *population* of proofs. It seeds **N = 32** candidates, verifies each with **K_verify = 4** samples, and then iterates for up to **R = 10** rounds. Each round selects the top **M = 4** diverse, non-perfect parents by fitness and applies two refinement operators: **PATCH**, an exploitation move that fixes specific errors while preserving the correct parts, and **REWRITE**, an exploration move that tries an entirely different proof route.

The search stops early only when **at least two** candidates independently reach a perfect 7/7 — a population-level signal that resists a single over-confident grade. Final answer selection is a pairwise tournament among the top **K = 4** survivors, decided by **K_ranker = 3** ranker votes. This is what converts a flaky best@K into a dependable pass@1.

![The MaxProof test-time scaling loop: initialize, verify, select parents, PATCH/REWRITE, and select by tournament](flow.png){.post-cover}

## Does it work?

On contest problems the test-time search adds a large, consistent boost over one-shot generation:

| Contest | M3 one-shot | M3 + MaxProof | Gain |
|---|---|---|---|
| IMO 2025 | 27 / 42 | 35 / 42 | +8 |
| USAMO 2026 | 26 / 42 | 36 / 42 | +10 |

On standalone one-shot benchmarks M3 scores 67.40 on IMOProofBench and 81.56 on IMOAnswerBench — behind GPT-5.5 (90.85 / 90.60) and Gemini 3.1 Pro (75.71 / 90.00), confirming the "follower" framing. But the per-problem analysis is telling: 9 of 12 contest problems reached the oracle-best 7/7 by round 4. Of the three that didn't, one hit a genuine capability ceiling (IMO P6, a missing core idea) while the other two were *selection* losses — the tournament left a 6/7 proof sitting in the archive and shipped a worse one — pointing squarely at verifier calibration at the correctness boundary as the next bottleneck.

A recurring qualitative note: the proofs that earn full marks tend to win through exhaustive, defensive case analysis rather than elegant insight — a direct reflection of the conservative, false-negative-favoring training signal.

::: {.callout-note title="Key Result"}
Population-level test-time scaling lifts M3 from **27→35 on IMO 2025** and **26→36 on USAMO 2026** (+8 and +10 problems), showing that a reliable verifier plus inference-time search can narrow the gap to stronger closed-source systems through system design rather than scale alone.
:::

## Takeaway

MaxProof reframes proof solving from a one-shot generation problem into an *evolving process* — propose, challenge, repair, and choose, with increasing discipline. Its core lesson generalizes well beyond mathematics: when your reward has no executable oracle, the verifier *is* the product. Invest in making it trustworthy (defense-in-depth, pessimistic aggregation, harvested error data), keep your experts anchored to that shared ground truth, and spend test-time compute on a population rather than a point. The remaining frontier — missing core ideas and verifier miscalibration at the boundary — is exactly where the authors point future work.